# Bloco 1: Coleta de dados legislativos sobre Inteligência Artificial
**Minicurso "IA como Instrumento de Medição": II DataPOL (UFPE, 2026)**

Câmara dos Deputados + Senado Federal, 2019–2026, via APIs de dados abertos.

**Pergunta de pesquisa do minicurso:**
> *Quando o Congresso legisla sobre IA, ele quer PUNIR, REGULAR ou PROMOVER? Isso mudou com as eleições (2022, 2024, 2026) e difere entre campos políticos?*

Saída: `dados/processado/corpus.csv`. A coleta completa leva de 5 a 10 minutos (são centenas de chamadas às APIs). Em sala usamos o arquivo já salvo; este notebook documenta **como** ele foi gerado (reprodutibilidade).

In [1]:
# === Configuração: funciona localmente e no Google Colab ===
import os, sys, subprocess
from pathlib import Path

# PROVISÓRIO: endereço do repositório público do minicurso (ajustar quando publicado)
REPO_URL = "https://github.com/jodavid/minicurso-ia-medicao"

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    if not Path("minicurso-ia-medicao").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    RAIZ = Path("minicurso-ia-medicao").resolve()
else:
    # localmente: a raiz é a pasta do minicurso (este notebook fica em python/)
    RAIZ = Path.cwd().parent if Path.cwd().name == "python" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "python"))
SAIDA = RAIZ / "resultados" / "python"          # o R grava em resultados/; o Python, aqui
(SAIDA / "figuras").mkdir(parents=True, exist_ok=True)
(SAIDA / "tabelas").mkdir(parents=True, exist_ok=True)
print("Raiz do projeto:", RAIZ)

Raiz do projeto: ~/minicurso_IA_medicao_publico


In [2]:
import pandas as pd
from datetime import date
from medicao import (camara_proposicoes, camara_autor, camara_situacao, camara_partido_na_data,
                     senado_processos, extrai_partido_senado, limpa_ementa, campo_politico)

## Parâmetros

`MODO_TESTE = True` coleta só um termo e um ano (rápido, para ver o mecanismo funcionando) e salva em `resultados/python/`.
Com `MODO_TESTE = False`, a coleta é completa e **sobrescreve** `dados/processado/corpus.csv`.

In [3]:
MODO_TESTE = True

TERMOS = ["inteligência artificial", "deepfake", "deep fake", "conteúdo sintético"]
ANOS = list(range(2019, 2027))
if MODO_TESTE:
    TERMOS, ANOS = ["deepfake"], [2024]
DATA_COLETA = date.today().isoformat()

## 1. Câmara dos Deputados

A API limita intervalos de datas a 3 meses; por isso filtramos por `ano`, que não tem essa restrição.

In [4]:
camara = pd.concat([camara_proposicoes(t, a) for t in TERMOS for a in ANOS], ignore_index=True)
camara = camara.drop_duplicates("id")
print(len(camara), "proposições na Câmara")

autores = pd.DataFrame([camara_autor(i) for i in camara["id"]])
situacao = pd.DataFrame([camara_situacao(i) for i in camara["id"]])
camara = camara.merge(autores, on="id", how="left").merge(situacao, on="id", how="left")

6 proposições na Câmara


### Partido **na data** da apresentação

Detalhe que muda a medida e costuma ser ignorado: o partido atual do deputado não é o partido que ele tinha quando apresentou o projeto. Além disso, o histórico da API traz registros "Partido no início da legislatura" com data de 2023-02-01 para **todas** as legislaturas anteriores (com `situacao` nula). Sem filtrá-los, um deputado do PL em 2023 aparece como PMDB ou PFL.

In [5]:
partidos = pd.DataFrame([camara_partido_na_data(u, d)
                         for u, d in zip(camara["uri_autor"], camara["data_apresentacao"])])
camara = pd.concat([camara.reset_index(drop=True), partidos], axis=1)
camara.head()

,id,sigla,numero,ano,ementa,data_apresentacao,autor,tipo_autor,uri_autor,n_autores,situacao,data_situacao,partido,uf
0,2422375,PL,896,2024,Dispõe sobre a Proteção contra Deepfakes e dá ...,2024-03-20T15:46,Saullo Vianna,Deputado(a),https://dadosabertos.camara.leg.br/api/v2/depu...,1,Tramitando em Conjunto,2025-07-28T10:58,UNIÃO,AM
1,2424878,PL,1119,2024,"Altera a Lei nº 12.965, de 23 de abril de 2014...",2024-04-08T16:15,Emanuel Pinheiro Neto,Deputado(a),https://dadosabertos.camara.leg.br/api/v2/depu...,1,Tramitando em Conjunto,2024-04-18T17:12,MDB,MT
2,2445631,PL,2709,2024,"Altera a Lei n.º 8.069, de 12 de julho de 1990...",2024-07-03T15:55,Rogéria Santos,Deputado(a),https://dadosabertos.camara.leg.br/api/v2/depu...,1,Aguardando Apreciação pelo Senado Federal,2025-12-19T00:00,REPUBLICANOS,BA
3,2457308,PL,3488,2024,"Altera a Lei nº 12.965, de 23 de abril de 2014...",2024-09-09T20:27,Alexandre Guimarães,Deputado(a),https://dadosabertos.camara.leg.br/api/v2/depu...,1,Aguardando Designação de Relator(a),2025-05-15T00:00,MDB,TO
4,2467209,PL,4272,2024,Altera o Estatuto da Criança e do Adolescente ...,2024-11-05T15:59,Rubens Pereira Júnior,Deputado(a),https://dadosabertos.camara.leg.br/api/v2/depu...,1,Tramitando em Conjunto,2025-03-19T17:58,PT,MA


## 2. Senado Federal

No Senado, a autoria já vem com partido e UF na data ("Senador Fulano (PSD/MG)").

In [6]:
lst = [x for x in (senado_processos(t, a) for t in TERMOS for a in ANOS) if len(x)]
senado = pd.concat(lst, ignore_index=True).drop_duplicates("id") if lst else pd.DataFrame()
print(len(senado), "processos no Senado")

0 processos no Senado


## 3. Corpus unificado

In [7]:
camara_pad = pd.DataFrame(dict(
    casa="Câmara",
    identificacao=camara["sigla"] + " " + camara["numero"].astype(str) + "/" + camara["ano"].astype(str),
    ano=camara["ano"], data_apresentacao=camara["data_apresentacao"].str[:10],
    ementa=camara["ementa"], autor=camara.get("autor"), tipo_autor=camara.get("tipo_autor"),
    partido=camara["partido"], uf=camara["uf"], situacao=camara["situacao"]))

if len(senado):
    ps = senado["autoria"].apply(extrai_partido_senado)
    senado_pad = pd.DataFrame(dict(
        casa="Senado", identificacao=senado["identificacao"],
        ano=senado["data_apresentacao"].str[:4].astype(int),
        data_apresentacao=senado["data_apresentacao"], ementa=senado["ementa"],
        autor=senado["autoria"].str.replace(r"\s*\(.*\)$", "", regex=True),
        tipo_autor=senado["autoria"].fillna("").str.startswith("Senador").map({True: "Senador(a)", False: "Outro"}),
        partido=[p for p, _ in ps], uf=[u for _, u in ps], situacao=senado["situacao"]))
else:
    senado_pad = pd.DataFrame(columns=camara_pad.columns)

# Projetos do Senado remetidos à Câmara aparecem nas duas casas (ex.: PL 2338/2023).
# Mantemos o registro da casa de origem (Senado vem primeiro), que traz o partido na data.
corpus = (pd.concat([senado_pad, camara_pad], ignore_index=True)
          .drop_duplicates("identificacao")
          .assign(ementa=lambda d: d["ementa"].map(limpa_ementa),
                  partido=lambda d: d["partido"].str.strip().str.upper()))
corpus = corpus[corpus["ano"].isin(ANOS) & (corpus["ementa"].str.len() > 20)]
corpus = corpus.sort_values("data_apresentacao").reset_index(drop=True)
corpus.insert(0, "doc_id", [f"doc{i:04d}" for i in range(1, len(corpus) + 1)])

## 4. Campo político

Simplificação didática a partir de Bolognesi, Ribeiro & Codato (2023), *A New Ideological Classification of Brazilian Political Parties*, Dados 66(2). Ela própria é uma **medida**, com erro e escolhas; discutimos isso em sala.

In [8]:
corpus["campo"] = corpus["partido"].map(campo_politico)
corpus["ano_eleitoral"] = corpus["ano"] % 2 == 0      # 2020, 2022, 2024, 2026
corpus["data_coleta"] = DATA_COLETA

destino = (SAIDA / "corpus_teste.csv") if MODO_TESTE else (RAIZ / "dados/processado/corpus.csv")
corpus.to_csv(destino, index=False)
print(f"Corpus: {len(corpus)} proposições salvas em {destino}")
corpus.groupby(["casa", "ano"]).size()

Corpus: 6 proposições salvas em ~/minicurso_IA_medicao_publico/resultados/python/corpus_teste.csv


casa    ano 
Câmara  2024    6
dtype: int64

In [9]:
corpus[["doc_id", "casa", "identificacao", "partido", "campo", "ementa"]].head(10)

,doc_id,casa,identificacao,partido,campo,ementa
0,doc0001,Câmara,PL 896/2024,UNIÃO,Direita,Dispõe sobre a Proteção contra Deepfakes e dá ...
1,doc0002,Câmara,PL 1119/2024,MDB,Centro,"Altera a Lei nº 12.965, de 23 de abril de 2014..."
2,doc0003,Câmara,PL 2709/2024,REPUBLICANOS,Direita,"Altera a Lei n.º 8.069, de 12 de julho de 1990..."
3,doc0004,Câmara,PL 3488/2024,MDB,Centro,"Altera a Lei nº 12.965, de 23 de abril de 2014..."
4,doc0005,Câmara,PL 4272/2024,PT,Esquerda,Altera o Estatuto da Criança e do Adolescente ...
5,doc0006,Câmara,PL 4768/2024,UNIÃO,Direita,Dispõe sobre a responsabilização do crime de m...
